# Comparison of Statistical POS Taggers – Averaged Perceptron Classifier

## Introduction

This notebook is part of a series of three experiments comparing different **statistical part-of-speech taggers** using the **Brown Corpus**.  
Each notebook focuses on a specific tagging approach; this one evaluates the **Averaged Perceptron classifier** as implemented in NLTK.

The workflow includes:
- training the tagger on a subset of the Brown corpus,
- predicting POS tags on unseen data,
- evaluating performance using standard classification metrics,
- exporting sample results for inspection.

## References on the Averaged Perceptron Tagger
- **[Berger 1996]** Berger, A. L.; Della Pietra, S. A.; Della Pietra, V. J. (1996).  
  *A Maximum Entropy Approach to Natural Language Processing.*  
  Computational Linguistics, 22, pp. 39–71.

- **[Brants 2000]** Brants, T. (2000).  
  *TnT – A Statistical Part-of-Speech Tagger.*  
  Proceedings of ANLP 2000, Seattle, WA, pp. 224–231.

## Imports and Dependencies

We import standard Python libraries for timing, file export, and evaluation,
as well as NLTK modules required for corpus access, tagging, and tagset mapping.

In [ ]:
import nltk

# Ensure Brown corpus is available
try:
    nltk.data.find("corpora/brown")
except LookupError:
    nltk.download("brown")

In [2]:
import csv
import time
import datetime

from sklearn import metrics

from nltk.tag import map_tag
from nltk.tag.perceptron import PerceptronTagger
from nltk.corpus import brown as corpus

## Dataset Split and Experimental Setup

The corpus is split into training and test partitions, with **80% of the documents used for training** and the remaining 20% reserved for evaluation.

Files are taken in their original order, and **no shuffling is applied**.

> ⚠️ **Important note on execution time**  
 Training the Averaged Perceptron on a large corpus such as Brown is **computationally expensive**.  
 Depending on hardware resources (CPU speed, memory, Python environment), this step can take **tens of minutes and in some cases close to one hour** to complete.  This is due to the large training corpus and to the fact that the process is CPU-bound and does not benefit from GPU acceleration.

> 💡 **Tip for faster experimentation**
To significantly reduce training time during testing or demonstration, set `training_length` to a **smaller number of files** (for example, 5–20) using cell "OPTIONAL ALTERNATIVE SPLIT).
This allows quicker iteration at the cost of lower tagging accuracy.

> **Results for work *Tarea del tema 4: Comparativa de etiquetadores estadísticos***
To reproduce results of this work run the optional cell split, setting `training length` = 105 and `testing length` = 50


In [3]:
# ======================
# DATASET SPLIT
# ======================

# Total number of files in the Brown corpus
corpus_length = len(corpus.fileids())
print(f"Total corpus files: {corpus_length}")

# Define training size as 80% of the corpus
# NOTE: Reducing this value (e.g. training_length = 50) 
# will significantly speed up training for experimentation
# Use alternative cell for this option.
training_length = int(corpus_length * 0.8)
print(f'Training length: {training_length} files')
testing_length = int(corpus_length * 0.2)
print(f'Testing length: {testing_length} files')

Total corpus files: 500
Training length: 400 files
Testing length: 100 files


In [4]:
# ======================
# OPTIONAL ALTERNATIVE SPLIT (NOT USED BY DEFAULT)
# ======================
# This block illustrates how the dataset size could be customized.
# It is NOT executed unless the user edits the values below.

# User-defined sizes (must satisfy: train + test <= corpus_length)
# training_length = 105
# testing_length = 50
# print(f'New training length: {training_length} files')
# print(f'New testing length: {testing_length} files')

# Uncomment to activate:
# training_files = corpus.fileids()[:training_length]
# testing_files = corpus.fileids()[training_length:training_length + testing_length]

## Training

Training is performed **from scratch** (load=False), meaning that no pre-trained weights are used. The total training time is measured to provide an indication of the computational cost of this approach on standard hardware.

In [5]:
# ======================
# TRAINING PHASE
# ======================
# This section trains an Averaged Perceptron POS tagger using
# a subset of the Brown Corpus.

# Load tagged sentences from the selected training files
# Each sentence is a list of (word, tag) tuples
training_sentences = corpus.tagged_sents(
    corpus.fileids()[0:training_length]
)

# Initialize the Averaged Perceptron tagger
# load=False ensures the model is trained from scratch
tagger = PerceptronTagger(load=False)
print(f'Tagger initialized: {tagger}')

# Measure training time
start = time.time()

# Train the tagger on the training sentences
tagger.train(training_sentences)

end = time.time()

# Compute total training time
processing_time = end - start
print(f'Training performed in {processing_time:.2f} seconds')

Tagger initialized: <nltk.tag.perceptron.PerceptronTagger object at 0x000001CA19BD4590>
Training performed in 1971.53 seconds


## Brief Qualitative Evaluation on Unseen Data

This section provides a **qualitative inspection** of the tagger’s behavior on **unseen data**.  

A single test document from the Brown corpus—excluded from training—is selected to allow a direct comparison between the **gold-standard POS annotations** and the **POS tags predicted** by the trained Averaged Perceptron tagger.

In [ ]:
# BRIEF QUALITATIVE ANALYSIS
# Inspection of gold vs. predicted POS tags on a single unseen test document

# Select one test file (the first file after the training split)
test_file_id = corpus.fileids()[training_length]

# Retrieve gold-standard tagged sentences from the test file
qual_testing_sentences  = corpus.tagged_sents(test_file_id)

print(f"Qualitative evaluation on test file: {test_file_id}")
print("\nGold-standard labels (first tokens of first sentences):")
print(qual_testing_sentences[0][:5], qual_testing_sentences [1][:5])

# Remove POS tags to obtain raw word sequences
qual_bare_sentences = [[word for word, _ in sentence] for sentence in qual_testing_sentences ]
# Alternative:
# qual_bare_sentences = corpus.sents(test_file_id)

print("\nRaw words:")
print(qual_bare_sentences[0][:5], qual_bare_sentences[1][:5])

# Predict POS tags using the trained Perceptron tagger
qual_predicted_sentences = [tagger.tag([word for word,_ in sentence]) for sentence in qual_testing_sentences ]

print("\nPredicted labels (tagger default's tagset):")
print(qual_predicted_sentences[0][:5], qual_predicted_sentences[1][:5])


Qualitative evaluation on test file: ck27

Gold-standard labels (first tokens of first sentences):
[('``', '``'), ('But', 'CC'), ('tell', 'VB'), ('me', 'PPO'), (',', ',')] [('Alex', 'NP'), ('asked', 'VBD'), ('.', '.')]

Raw words:
['``', 'But', 'tell', 'me', ','] ['Alex', 'asked', '.']

Predicted labels (tagger default's tagset):
[('``', '``'), ('But', 'CC'), ('tell', 'VB'), ('me', 'PPO'), (',', ',')] [('Alex', 'NP'), ('asked', 'VBD'), ('.', '.')]


## Quantitative Evaluation

This section provides a **quantitative evaluation** of the Averaged Perceptron tagger.

The test set consists of the **50 files from the Brown corpus ** that were **not used during training**.  
All sentences from these files are evaluated jointly in order to compute **global performance metrics**.

Gold-standard POS tags and predicted tags are flattened into single lists, allowing the computation of:
- accuracy,
- precision,
- recall,
- and F1-score,

using weighted averages to account for class imbalance.

In [7]:
# QUANTITATIVE EVALUATION
# Global performance metrics on the test set (20% of Brown corpus files)

# Load tagged sentences from the selected testing files
testing_sentences = corpus.tagged_sents(
    corpus.fileids()[training_length : training_length + testing_length]
)

# Predict POS tags on the whole test set
predicted_sentences = [
    tagger.tag([word for word, _ in sentence])
    for sentence in testing_sentences
]

# Flatten gold-standard POS tags from all test sentences
golds = [tag for sentence in testing_sentences for _,tag in sentence]

# Flatten predicted POS tags
predicted_labels = [tag for sentence in predicted_sentences for _,tag in sentence]

print("Gold labels (sample):")
print(golds[:5])

print("\nPredicted labels (sample):")
print(predicted_labels[:5])

# Compute evaluation metrics
# Weighted averages account for tag frequency imbalance
accuracy = metrics.accuracy_score(golds, predicted_labels)
precision = metrics.precision_score(golds, predicted_labels, average="weighted", zero_division=0)
recall = metrics.recall_score(golds, predicted_labels, average="weighted", zero_division=0)
f1_score = metrics.f1_score(golds, predicted_labels, average="weighted", zero_division=0)
        
print("\nPerformance metrics for Averaged Perceptron Tagger:")
print(" Accuracy :", accuracy)
print(" Precision:", precision)
print(" Recall   :", recall)
print(" F1-Score :", f1_score)

Gold labels (sample):
['``', 'CC', 'VB', 'PPO', ',']

Predicted labels (sample):
['``', 'CC', 'VB', 'PPO', ',']

Performance metrics for Averaged Perceptron Tagger:
 Accuracy : 0.9583524466425909
 Precision: 0.9577697866477537
 Recall   : 0.9583524466425909
 F1-Score : 0.9573797378012883


## Exporting Sample Annotations for Qualitative Comparison

To support a **qualitative comparison among different POS taggers**, this section exports a small sample of annotated data from the test set.

The first **15 sentences** of the selected test set are extracted, and for each token the following information is stored:
- the word form,
- the gold-standard POS tag from the Brown corpus,
- the POS tag predicted by the Averaged Perceptron tagger.

Results are exported to a CSV file.

In [8]:
# ======================
# EXPORT FOR QUALITATIVE COMPARISON
# ======================
# Export gold vs. predicted POS tags for qualitative comparison 
# across different taggers.
#
# Note:
# Each row in the CSV corresponds to a single token.
# Sentence boundaries are implicit and can be reconstructed
# from the original sentence order.
    
# Timestamp for unique file naming
now = datetime.datetime.now().strftime("%y%m%d-%H%M")

tagger_name = 'Perceptron'
corpus_name = 'Brown'

# Output filename
output_filename = f"{now}-{tagger_name}-{corpus_name}-samples.csv"

# Number of sentences to export
n_sentences = 15

with open(output_filename, mode='w', newline='', encoding='utf-8') as dades:
    writer = csv.writer(dades, delimiter=',')

    # (Optional) header row for clarity
    writer.writerow(["word", "gold_tag", "predicted_tag"])

    # Iterate over the first n_sentences of the test set
    for i in range(min(n_sentences, len(testing_sentences))):
        for j in range(len(testing_sentences[i])):
            # Gold annotation: (word, gold_tag)
            word, gold_tag = testing_sentences[i][j]

            # Predicted annotation: (word, predicted_tag)
            _, predicted_tag = predicted_sentences[i][j]

            # Write combined row: word, gold_tag, predicted_tag
            writer.writerow((word, gold_tag, predicted_tag))

print("\nGold vs predicted labels (first token):")
print(
    testing_sentences[0][0][0],
    testing_sentences[0][0][1],
    predicted_sentences[0][0][1],
)
print(f"\nExported to: {output_filename}")



Gold vs predicted labels (first token):
`` `` ``

Exported to: 260224-1258-Perceptron-Brown-samples.csv
